# CS781 Phase 7

In [1]:
import os, math, json, time, random, zipfile, csv, gc, shutil, pickle
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.checkpoint import checkpoint

# ============================ CONFIG ============================
TECH_LIST        = os.environ.get("PCR_TECHS", "full,lorma,kd,partial,ia3,prompt").split(",")   # order matters: full before kd
TRAIN_MINUTES    = float(os.environ.get("PCR_MIN", 15))  # wall-clock training budget PER TECHNIQUE
MAX_CHUNKS       = 8        # chunks per document at inference (8*~512 tokens ~ 1.5k words)
MAX_WORDS        = 2400     # hard cap on words read per document (speed)
BATCH_Q          = 8        # queries per step
N_HARD           = 2        # hard negatives per query
TEMP             = 0.05     # contrastive temperature
POOLING          = "auto"   # "mean" | "first" | "auto" (pick better one on dev)
USE_TFIDF_FUSION = False    # hybrid dense+TF-IDF. Keep False unless TAs allow non-neural components.
SEED             = 42
LR = {"full": 5e-5, "partial": 1e-4, "lorma": 3e-4, "ia3": 3e-3, "prompt": 1e-2, "prefix": 3e-3, "qlora": 3e-4, "kd": 5e-5}
PARTIAL_LAYERS   = 3        # partial FT: last N blocks + final norm
LORMA_RANK       = 16
LORMA_ALPHA      = 16       # scale = alpha/rank
QLORA_RANK, QLORA_ALPHA, QBLOCK = 16, 32, 64   # LoRA rank/alpha on a 4-bit NF4 frozen base, quant block size
N_PREFIX         = 16       # prefix-tuning: learned key/value vectors per layer
N_PROMPT         = 32       # prompt tokens (prompt tuning)
KD_LAYERS        = [0, 2, 4, 6, 8, 11]   # student = these teacher layers (6-layer, ~half size)
KD_TEMP, KD_W, KD_COS_W = 0.05, 1.0, 0.5
MODEL_OVERRIDE   = None     # e.g. "/kaggle/input/.../model.pt"  (MUST be the 45k-step decoder)
TOK_OVERRIDE     = None
HF_REPO          = "Exploration-Lab/CS-781-Capstone"
HF_PREFIX        = "Phase_7_PCR"
OUT              = os.environ.get("PCR_OUT", "/kaggle/working/pcr")
DATA_DIR         = os.environ.get("PCR_DATA_DIR")      # local override (used for smoke tests)
# ================================================================
assert all(t in ["zeroshot", "full", "partial", "lorma", "ia3", "prompt", "prefix", "qlora", "kd"] for t in TECH_LIST)
assert "kd" not in TECH_LIST or "full" in TECH_LIST[:TECH_LIST.index("kd")] or os.path.exists(f"{OUT}/enc_full.pt"), "kd needs full before it"
os.makedirs(OUT, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", DEVICE, "| techniques", TECH_LIST, "| train minutes each", TRAIN_MINUTES)

def find_files(name, root="/kaggle/input"):
    hits = []
    for dp, _, fns in os.walk(root, followlinks=True):
        if name in fns: hits.append(os.path.join(dp, name))
    return sorted(hits)

device cuda | techniques ['full', 'lorma', 'kd', 'partial', 'ia3', 'prompt'] | train minutes each 15.0


In [2]:
class ByteLevelBPETokenizer:
    SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}
        self.id_to_bytes = {i: bytes([i]) for i in range(256)}
        self.special_to_id = {}
        for i, tok in enumerate(self.SPECIAL_TOKENS):
            tid = 256 + i
            self.special_to_id[tok] = tid
            self.id_to_bytes[tid] = tok.encode("utf-8")
    @property
    def vocab_size(self): return len(self.id_to_bytes)
    @staticmethod
    def _apply_merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    def encode(self, text):
        ids = []
        for word in text.split():
            seq = list(word.encode("utf-8"))
            while len(seq) > 1:
                mergeable = [p for p in zip(seq, seq[1:]) if p in self.merges]
                if not mergeable: break
                pair = min(mergeable, key=lambda p: self.merges[p])
                seq = self._apply_merge(seq, pair, self.merges[pair])
            ids.extend(seq); ids.append(self.special_to_id["</s>"])
        return ids
    @classmethod
    def load(cls, path):
        tok = cls()
        data = json.load(open(path, "r", encoding="utf-8"))
        for pair_str, nid in data["merges"].items():
            a, b = (int(x) for x in pair_str.split(","))
            tok.merges[(a, b)] = nid
            tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__(); self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        return self.weight * x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)

def rotate_half(x):
    x1 = x[..., ::2]; x2 = x[..., 1::2]
    return torch.stack((-x2, x1), dim=-1).flatten(-2)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos", emb.cos()[None, None, :, :], persistent=False)
        self.register_buffer("sin", emb.sin()[None, None, :, :], persistent=False)
    def forward(self, q, k):
        T = q.size(-2); cos = self.cos[:, :, :T, :]; sin = self.sin[:, :, :T, :]
        return (q * cos) + (rotate_half(q) * sin), (k * cos) + (rotate_half(k) * sin)

class MultiQueryCausalAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, max_seq_len, dropout=0.0, rope_base=10000.0):
        super().__init__()
        self.n_heads, self.kv_heads = n_heads, kv_heads
        self.d_head = d_model // n_heads; self.kv_dim = kv_heads * self.d_head
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.v_proj = nn.Linear(d_model, self.kv_dim, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.rope = RotaryEmbedding(self.d_head, max_seq_len=max_seq_len, base=rope_base)
        self.dropout = dropout; self.resid_dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.kv_heads, self.d_head).transpose(1, 2)
        q, k = self.rope(q, k)
        if self.kv_heads != self.n_heads:
            r = self.n_heads // self.kv_heads
            k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=None,
                                             dropout_p=self.dropout if self.training else 0.0, is_causal=True)
        return self.resid_dropout(self.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate = nn.Linear(d_model, d_ff, bias=False); self.up = nn.Linear(d_model, d_ff, bias=False)
        self.down = nn.Linear(d_ff, d_model, bias=False); self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        return self.dropout(self.down(F.silu(self.gate(x)) * self.up(x)))

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base):
        super().__init__()
        self.norm_attn = RMSNorm(d_model)
        self.attn = MultiQueryCausalAttention(d_model, n_heads, kv_heads, max_seq_len, dropout, rope_base)
        self.norm_ffn = RMSNorm(d_model); self.ffn = SwiGLU(d_model, d_ff, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm_attn(x)); return x + self.ffn(self.norm_ffn(x))

class RoPEMQADecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, kv_heads, d_ff, max_seq_len, dropout=0.1,
                 rope_base=10000.0, tie_weights=True):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, kv_heads, d_ff, max_seq_len, dropout, rope_base)
                                     for _ in range(n_layers)])
        self.final_norm = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        if tie_weights: self.lm_head.weight = self.token_emb.weight

## Load Phase 1 checkpoint

In [3]:
if MODEL_OVERRIDE: model_path = MODEL_OVERRIDE
else:
    cands = find_files("model.pt"); assert cands, "No model.pt under /kaggle/input -> attach your decoder output as an Input"
    best_step, model_path = -1, cands[0]
    for m in cands:                                   # pick the most-trained decoder via the sibling checkpoint.pt
        ck = os.path.join(os.path.dirname(m), "checkpoint.pt")
        step = torch.load(ck, map_location="cpu", weights_only=False).get("step", -1) if os.path.exists(ck) else -1
        print(f"candidate {m} | training steps = {step}")
        if step > best_step: best_step, model_path = step, m
    print(">>> USING", model_path, "| steps =", best_step, "(set MODEL_OVERRIDE to force another)")
    print(">>> Report this step count in your write-up: Phase 7 starts from the %s-step decoder" % best_step)
tok_path = TOK_OVERRIDE or (find_files("bpe_tokenizer.json") or [None])[0]
assert tok_path, "bpe_tokenizer.json not found (attach it, or set TOK_OVERRIDE)"
ckpt = torch.load(model_path, map_location="cpu", weights_only=False)
CFG = ckpt["config"]; print("config:", CFG)
tokenizer = ByteLevelBPETokenizer.load(tok_path)
assert tokenizer.vocab_size == CFG["vocab_size"], (tokenizer.vocab_size, CFG["vocab_size"])
BLOCK = CFG["max_seq_len"]; PAD_ID = tokenizer.special_to_id["<pad>"]; EOS_ID = tokenizer.special_to_id["</s>"]
D = CFG["d_model"]
CHUNK = BLOCK - N_PROMPT        # tokens per chunk, same for every technique (prompt tuning's tokens eat into the window)

def build_lm(n_layers=None):
    return RoPEMQADecoderLM(CFG["vocab_size"], CFG["d_model"], n_layers or CFG["n_layers"], CFG["n_heads"],
                            CFG["kv_heads"], CFG["d_ff"], CFG["max_seq_len"], CFG["dropout"], CFG["rope_base"])

def load_phase1_lm(layer_ids=None):
    sd = ckpt["state_dict"]
    if layer_ids is None:
        lm = build_lm(); lm.load_state_dict(sd); return lm
    lm = build_lm(len(layer_ids)); new = {}
    for k, v in sd.items():
        if k.startswith("blocks."):
            parts = k.split("."); li = int(parts[1])
            if li in layer_ids: parts[1] = str(layer_ids.index(li)); new[".".join(parts)] = v
        else: new[k] = v
    lm.load_state_dict(new); return lm

candidate /kaggle/input/datasets/parth1223/decode-part-2/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt | training steps = 15000
candidate /kaggle/input/datasets/parth1223/decoder-output/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt | training steps = 7500
candidate /kaggle/input/notebooks/parth1223/decoder-part-2/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt | training steps = 15000
candidate /kaggle/input/notebooks/parth1223/decoder-part-3/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt | training steps = 45000
>>> USING /kaggle/input/notebooks/parth1223/decoder-part-3/cs781_phase1/rope_mqa_swiglu_train_supreme_court-train_acts-train_high_courts-train_tribunals/model.pt | steps = 45000 (set MODEL_OVERRIDE to force another)
>>> Report this step count in your write

## Encoder wrapper, adapters (LoRMA / IA3 / prompt tuning)

In [4]:
class PCREncoder(nn.Module):
    """ids -> mean-pooled, L2-normalised embedding. Right-padding is safe: attention is causal."""
    def __init__(self, lm, n_prompt=0):
        super().__init__()
        self.lm, self.n_prompt, self.grad_ckpt = lm, n_prompt, False
        if n_prompt: self.prompt = nn.Parameter(torch.randn(n_prompt, lm.token_emb.embedding_dim) * 0.02)
    def forward(self, ids, mask):
        x = self.lm.token_emb(ids)
        if self.n_prompt:
            x = torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], dim=1)
            mask = torch.cat([mask.new_ones(mask.size(0), self.n_prompt), mask], dim=1)
        for blk in self.lm.blocks:
            x = checkpoint(blk, x, use_reentrant=False) if (self.grad_ckpt and self.training) else blk(x)
        x = self.lm.final_norm(x)
        m = mask.unsqueeze(-1).to(x.dtype)
        e = (x * m).sum(1) / m.sum(1).clamp(min=1)
        return F.normalize(e.float(), dim=-1)

class LoRMALinear(nn.Module):
    """LoRMA (Bihany et al. 2025): MULTIPLICATIVE low-rank update W' = M W0 with M low-rank, applied on the OUTPUT side
    of the frozen W0. Naive M = BA cannot start at identity (rank(I)=d), so rank is inflated additively:
    M = I + (alpha/r) B A  ->  y = h + (alpha/r) B(A h), h = W0 x, B=0 at init (so W'=W0 at step 0)."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Linear(base.out_features, rank, bias=False); self.B = nn.Linear(rank, base.out_features, bias=False)
        nn.init.normal_(self.A.weight, std=1.0 / rank); nn.init.zeros_(self.B.weight)
    def forward(self, x):
        h = self.base(x); return h + self.scale * self.B(self.A(h))

class LoRALinear(nn.Module):
    """Additive LoRA on top of a (possibly quantised) frozen linear: y = W x + (alpha/r) B A x."""
    def __init__(self, base, rank, alpha):
        super().__init__()
        self.base, self.scale = base, alpha / rank
        self.A = nn.Linear(base.in_features, rank, bias=False); self.B = nn.Linear(rank, base.out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5)); nn.init.zeros_(self.B.weight)
    def forward(self, x):
        return self.base(x) + self.scale * self.B(self.A(x.to(self.A.weight.dtype)))

NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
       -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]

class QuantLinear(nn.Module):
    """Frozen 4-bit NormalFloat (NF4) linear with blockwise absmax scaling (QLoRA-style), two codes packed per byte.
    Weights are dequantised on the fly each forward. Pure PyTorch (no bitsandbytes)."""
    def __init__(self, lin, block=QBLOCK):
        super().__init__()
        self.in_features, self.out_features, self.block = lin.in_features, lin.out_features, block
        w = lin.weight.data.float().flatten(); n = w.numel(); pad = (-n) % block
        wp = F.pad(w, (0, pad)).view(-1, block)
        absmax = wp.abs().amax(1, keepdim=True).clamp(min=1e-8)
        code = torch.tensor(NF4); idx = (wp / absmax).unsqueeze(-1).sub(code).abs().argmin(-1).to(torch.uint8).flatten()
        self.register_buffer("packed", (idx[0::2] << 4) | idx[1::2])
        self.register_buffer("absmax", absmax.squeeze(1).half()); self.register_buffer("code", code)
        self.n = n
    def dequant(self, dtype):
        hi, lo = (self.packed >> 4).long(), (self.packed & 15).long()
        idx = torch.stack([hi, lo], dim=1).flatten()
        w = self.code[idx].view(-1, self.block) * self.absmax.float().unsqueeze(1)
        return w.flatten()[:self.n].view(self.out_features, self.in_features).to(dtype)
    def forward(self, x):
        return F.linear(x, self.dequant(x.dtype))

class PrefixCausalAttention(nn.Module):
    """Prefix tuning: learned per-layer key/value vectors prepended to the attention memory (every token attends to them)."""
    def __init__(self, attn, n_prefix):
        super().__init__()
        self.a, self.P = attn, n_prefix
        self.pk = nn.Parameter(torch.randn(1, attn.kv_heads, n_prefix, attn.d_head) * 0.02)
        self.pv = nn.Parameter(torch.randn(1, attn.kv_heads, n_prefix, attn.d_head) * 0.02)
    def forward(self, x):
        a, P = self.a, self.P; B, T, C = x.shape
        q = a.q_proj(x).view(B, T, a.n_heads, a.d_head).transpose(1, 2)
        k = a.k_proj(x).view(B, T, a.kv_heads, a.d_head).transpose(1, 2)
        v = a.v_proj(x).view(B, T, a.kv_heads, a.d_head).transpose(1, 2)
        q, k = a.rope(q, k)
        k = torch.cat([self.pk.expand(B, -1, -1, -1).to(k.dtype), k], dim=2)
        v = torch.cat([self.pv.expand(B, -1, -1, -1).to(v.dtype), v], dim=2)
        if a.kv_heads != a.n_heads:
            r = a.n_heads // a.kv_heads; k = k.repeat_interleave(r, dim=1); v = v.repeat_interleave(r, dim=1)
        mask = torch.ones(T, P + T, dtype=torch.bool, device=x.device)
        mask[:, P:] = torch.tril(torch.ones(T, T, dtype=torch.bool, device=x.device))
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, dropout_p=a.dropout if self.training else 0.0)
        return a.resid_dropout(a.out_proj(out.transpose(1, 2).contiguous().view(B, T, C)))

class IA3Linear(nn.Module):
    """IA3: learned rescaling vector on the output (k, v) or input (FFN hidden) of a frozen linear."""
    def __init__(self, base, where):
        super().__init__()
        self.base, self.where = base, where
        self.vec = nn.Parameter(torch.ones(base.out_features if where == "out" else base.in_features))
    def forward(self, x):
        return self.base(x) * self.vec if self.where == "out" else self.base(x * self.vec)

def apply_technique(enc, tech):
    lm = enc.lm
    for p in enc.parameters(): p.requires_grad = False
    if tech in ("full", "kd"):
        for p in lm.parameters(): p.requires_grad = True
    elif tech == "partial":
        for blk in lm.blocks[-PARTIAL_LAYERS:]:
            for p in blk.parameters(): p.requires_grad = True
        for p in lm.final_norm.parameters(): p.requires_grad = True
    elif tech == "lorma":
        for blk in lm.blocks:
            for name in ("q_proj", "v_proj", "out_proj"):
                setattr(blk.attn, name, LoRMALinear(getattr(blk.attn, name), LORMA_RANK, LORMA_ALPHA))
    elif tech == "ia3":
        for blk in lm.blocks:
            blk.attn.k_proj = IA3Linear(blk.attn.k_proj, "out"); blk.attn.v_proj = IA3Linear(blk.attn.v_proj, "out")
            blk.ffn.down = IA3Linear(blk.ffn.down, "in")
    elif tech == "prompt":
        enc.prompt.requires_grad = True
    elif tech == "prefix":
        for blk in lm.blocks: blk.attn = PrefixCausalAttention(blk.attn, N_PREFIX)
    elif tech == "qlora":
        for blk in lm.blocks:                                     # 1) quantise every frozen linear to NF4
            for mod in (blk.attn, blk.ffn):
                for n, ch in list(mod.named_children()):
                    if isinstance(ch, nn.Linear): setattr(mod, n, QuantLinear(ch))
        for blk in lm.blocks:                                     # 2) LoRA adapters on the attention projections
            for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
                setattr(blk.attn, name, LoRALinear(getattr(blk.attn, name), QLORA_RANK, QLORA_ALPHA))
    # adapter params are created after freezing; make sure ONLY adapters train (base weights stay frozen)
    if tech in ("lorma", "ia3", "qlora", "prefix"):
        for n, p in enc.named_parameters():
            p.requires_grad = (".A." in n or ".B." in n or n.endswith(".vec") or n.endswith(".pk") or n.endswith(".pv"))
    return [p for p in enc.parameters() if p.requires_grad]

def count_params(m, trainable=False):
    seen, tot = set(), 0
    for p in m.parameters():
        if id(p) in seen or (trainable and not p.requires_grad): continue
        seen.add(id(p)); tot += p.numel()
    return tot

## Data (schema is auto-detected and printed — check it once)

In [5]:
def get_path(rel):
    if DATA_DIR: return os.path.join(DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    try:
        from kaggle_secrets import UserSecretsClient
        tokn = UserSecretsClient().get_secret("HuggingFace")
    except Exception: tokn = None
    return hf_hub_download(repo_id=HF_REPO, filename=f"{HF_PREFIX}/{rel}", repo_type="dataset", token=tokn)

def _clip_text(t):
    if isinstance(t, str): return " ".join(t.split()[:MAX_WORDS])
    words = []
    for s in t:                                    # list of sentences (possibly nested)
        words.extend((" ".join(s) if isinstance(s, list) else str(s)).split())
        if len(words) >= MAX_WORDS: break
    return " ".join(words[:MAX_WORDS])

def load_split(path, name):
    raw = json.load(open(path, "r", encoding="utf-8"))
    items = raw.items() if isinstance(raw, dict) else [(str(r.get("id", r.get("case_id", i))), r) for i, r in enumerate(raw)]
    out = {}
    for k, v in items:
        if isinstance(v, dict):
            text = v.get("text", v.get("document", "")); rel = v.get("relevant_candidates", v.get("relevant", v.get("labels", [])))
        else: text, rel = v, []
        out[str(k)] = {"text": _clip_text(text), "rel": [str(x) for x in (rel or [])]}
    ex = next(iter(out.items()))
    print(f"[{name}] n={len(out)} | e.g. id={ex[0]} words={len(ex[1]['text'].split())} n_rel={len(ex[1]['rel'])}")
    return out

SPL = {}
for split, qf, cf in [("train", "Train/train.json", "Train/train_candidates.json"),
                      ("dev", "Dev/dev.json", "Dev/dev_candidates.json"),
                      ("test", "Test/test.json", "Test/test_candidates.json")]:
    SPL[split] = {"q": load_split(get_path(qf), split + "_q"), "c": load_split(get_path(cf), split + "_cand")}
    if split != "test":
        pool = SPL[split]["c"]; nrel = [len([r for r in v["rel"] if r in pool]) for v in SPL[split]["q"].values()]
        print(f"   {split}: avg gold-in-pool/query={np.mean(nrel):.2f}, queries w/o gold in pool={sum(n == 0 for n in nrel)}")

Phase_7_PCR/Train/train.json: reconstructing file:   0%|          |  0.00B / 33.2MB            

Phase_7_PCR/Train/train.json: downloading bytes:           |  0.00B            

[train_q] n=827 | e.g. id=0000000599 words=2400 n_rel=2


Phase_7_PCR/Train/train_candidates.json: reconstructing file:   0%|          |  0.00B /  160MB            

Phase_7_PCR/Train/train_candidates.json: downloading bytes:           |  0.00B            

[train_cand] n=4320 | e.g. id=0000000262 words=2400 n_rel=0
   train: avg gold-in-pool/query=6.42, queries w/o gold in pool=10


... (6 outputs left out)

[test_q] n=237 | e.g. id=0000021652 words=2400 n_rel=0


Phase_7_PCR/Test/test_candidates.json: reconstructing file:   0%|          |  0.00B / 71.1MB            

Phase_7_PCR/Test/test_candidates.json: downloading bytes:           |  0.00B            

[test_cand] n=1727 | e.g. id=0000000987 words=2400 n_rel=0


## Tokenise (word-level memo -> fast) and chunk

In [6]:
MAX_TOKENS = MAX_CHUNKS * CHUNK
_memo = {}
def tok_doc(text):
    ids = []
    for w in text.split():
        t = _memo.get(w)
        if t is None: t = _memo[w] = tokenizer.encode(w)
        ids.extend(t)
        if len(ids) >= MAX_TOKENS: break
    return np.asarray(ids[:MAX_TOKENS] or [EOS_ID], dtype=np.int16)

TOK = {}
tcache = f"{OUT}/tok_cache_{CHUNK}_{MAX_CHUNKS}_{MAX_WORDS}.pkl"
if os.path.exists(tcache): TOK = pickle.load(open(tcache, "rb"))
else:
    t0 = time.time()
    for s in SPL:
        TOK[s] = {"q": {k: tok_doc(v["text"]) for k, v in SPL[s]["q"].items()},
                  "c": {k: tok_doc(v["text"]) for k, v in SPL[s]["c"].items()}}
    pickle.dump(TOK, open(tcache, "wb")); print("tokenised in %.0fs, memo=%d" % (time.time() - t0, len(_memo)))
lens = [len(a) for a in TOK["dev"]["c"].values()]
print("dev candidate tokens: mean %.0f, median %.0f, max %d (cap %d)" % (np.mean(lens), np.median(lens), max(lens), MAX_TOKENS))

def chunks_of(a): return [a[i:i + CHUNK] for i in range(0, len(a), CHUNK)]
def collate(arrs):
    L = max(len(a) for a in arrs)
    ids = torch.full((len(arrs), L), PAD_ID, dtype=torch.long); mask = torch.zeros(len(arrs), L, dtype=torch.long)
    for i, a in enumerate(arrs): ids[i, :len(a)] = torch.from_numpy(a.astype(np.int64)); mask[i, :len(a)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

tokenised in 9s, memo=208273
dev candidate tokens: mean 3498, median 3840, max 3840 (cap 3840)


## Embedding, scoring, Micro-F1@7

In [7]:
@torch.no_grad()
def embed_docs(enc, docs, bs=48):
    """Returns keys, E_first (first chunk only), E_mean (mean over chunks) — both L2-normalised, on CPU."""
    enc.eval(); keys = list(docs); items = []
    for ki, k in enumerate(keys):
        for c in chunks_of(docs[k]): items.append((ki, c))
    order = sorted(range(len(items)), key=lambda i: len(items[i][1]))
    emb = torch.zeros(len(items), D)
    for s in range(0, len(order), bs):
        idx = order[s:s + bs]; ids, mask = collate([items[i][1] for i in idx])
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            e = enc(ids, mask)
        emb[idx] = e.float().cpu()
    kid = torch.tensor([i for i, _ in items])
    msum = torch.zeros(len(keys), D).index_add_(0, kid, emb)
    first = torch.zeros(len(keys), D); seen = torch.zeros(len(keys), dtype=torch.bool)
    for j, (ki, _) in enumerate(items):
        if not seen[ki]: first[ki] = emb[j]; seen[ki] = True
    return keys, F.normalize(first, dim=-1), F.normalize(msum, dim=-1)

def micro_f1_at_k(pred, gold, k=7):
    hit = sum(len(set(pred[q][:k]) & set(gold[q])) for q in pred)
    npred, ngold = sum(len(pred[q][:k]) for q in pred), sum(len(gold[q]) for q in pred)
    p, r = hit / max(npred, 1), hit / max(ngold, 1)
    return {"P@7": p, "R@7": r, "F1@7": 2 * p * r / max(p + r, 1e-9)}

def topk_preds(S, qkeys, ckeys, k=7):
    S = S.clone(); cidx = {c: i for i, c in enumerate(ckeys)}
    for i, q in enumerate(qkeys):
        if q in cidx: S[i, cidx[q]] = -1e9                      # never retrieve the query itself
    top = S.topk(k, dim=1).indices.tolist()
    return {q: [ckeys[j] for j in top[i]] for i, q in enumerate(qkeys)}

def tfidf_matrix(qtexts, ctexts):
    from sklearn.feature_extraction.text import TfidfVectorizer
    v = TfidfVectorizer(sublinear_tf=True, min_df=2, max_df=0.5, max_features=300000, dtype=np.float32)
    C = v.fit_transform(ctexts); Q = v.transform(qtexts)
    return torch.from_numpy((Q @ C.T).toarray())

def gold_of(split):
    pool = SPL[split]["c"]
    return {q: [r for r in v["rel"] if r in pool] for q, v in SPL[split]["q"].items()}

def evaluate_split(enc, split):
    qk, qf, qm = embed_docs(enc, TOK[split]["q"]); ck, cf, cm = embed_docs(enc, TOK[split]["c"])
    gold = gold_of(split); res = {}
    for name, (Q, C) in {"first": (qf, cf), "mean": (qm, cm)}.items():
        res[name] = micro_f1_at_k(topk_preds(Q @ C.T, qk, ck), gold)
    return res, (qk, qf, qm, ck, cf, cm)

## TF-IDF hard-negative mining (train) + training loop

In [8]:
gold_tr = gold_of("train")
train_qs = [q for q, g in gold_tr.items() if g]
print("train queries with gold in pool:", len(train_qs), "/", len(gold_tr))
HARD = {}
if True:
    qk_ = list(SPL["train"]["q"]); ck_ = list(SPL["train"]["c"])
    S = tfidf_matrix([SPL["train"]["q"][q]["text"] for q in qk_], [SPL["train"]["c"][c]["text"] for c in ck_])
    for i, q in enumerate(qk_):
        g = set(gold_tr[q]); order = S[i].topk(min(60, S.size(1))).indices.tolist()
        HARD[q] = [ck_[j] for j in order if ck_[j] not in g and ck_[j] != q][:50]
    del S; gc.collect()

def rand_chunk(arr, rng):
    cs = chunks_of(arr); return cs[rng.randrange(len(cs))]

def make_batch(rng):
    qs = rng.sample(train_qs, min(BATCH_Q, len(train_qs)))
    pos = [rng.choice(gold_tr[q]) for q in qs]
    hn = [h for q in qs for h in rng.sample(HARD[q], N_HARD)] if N_HARD else []
    arrs = [rand_chunk(TOK["train"]["q"][q], rng) for q in qs] + \
           [rand_chunk(TOK["train"]["c"][c], rng) for c in pos + hn]
    cand_keys = pos + hn
    fmask = torch.zeros(len(qs), len(cand_keys), dtype=torch.bool)       # false negatives = other gold docs
    for i, q in enumerate(qs):
        g = set(gold_tr[q])
        for j, c in enumerate(cand_keys):
            if j != i and c in g: fmask[i, j] = True
    return arrs, len(qs), fmask.to(DEVICE)

def run_train(enc, params, lr, teacher=None):
    rng = random.Random(SEED); opt = torch.optim.AdamW(params, lr=lr, weight_decay=0.01)
    scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    enc.train(); enc.grad_ckpt = True; budget = TRAIN_MINUTES * 60; t0 = time.time(); step = 0; hist = []
    while True:
        frac = (time.time() - t0) / budget
        if frac >= 1: break
        cur = lr * (frac / 0.05 if frac < 0.05 else 0.5 * (1 + math.cos(math.pi * (frac - 0.05) / 0.95)))
        for g in opt.param_groups: g["lr"] = max(cur, lr * 0.02)
        arrs, nq, fmask = make_batch(rng); ids, mask = collate(arrs)
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            e = enc(ids, mask)
            q, c = e[:nq], e[nq:]
            logits = (q @ c.T / TEMP).masked_fill(fmask, -1e4)
            loss = F.cross_entropy(logits, torch.arange(nq, device=DEVICE)); parts = {"ce": loss.item()}
            if teacher is not None:
                with torch.no_grad():
                    te = teacher(ids, mask)
                tlog = (te[:nq] @ te[nq:].T / KD_TEMP).masked_fill(fmask, -1e4)
                slog = (q @ c.T / KD_TEMP).masked_fill(fmask, -1e4)
                kd = F.kl_div(F.log_softmax(slog, -1), F.softmax(tlog, -1), reduction="batchmean")
                cosl = (1 - (e * te).sum(-1)).mean()
                loss = loss + KD_W * kd + KD_COS_W * cosl; parts.update(kd=kd.item(), cos=cosl.item())
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); step += 1
        if step % 20 == 0:
            hist.append((step, loss.item())); print(f"step {step} t={time.time() - t0:.0f}s lr={cur:.2e} loss={loss.item():.4f}", {k: round(v, 4) for k, v in parts.items()})
    enc.grad_ckpt = False
    return step, time.time() - t0, hist

train queries with gold in pool: 817 / 827


## Run every technique: train -> dev eval -> test prediction -> zip (one function, looped)

In [9]:
import traceback

def run_technique(TECHNIQUE):
    t_start = time.time(); teacher = None
    if TECHNIQUE == "kd":
        tp = f"{OUT}/enc_full.pt"
        if not os.path.exists(tp): tp = (find_files("enc_full.pt") or [None])[0]
        assert tp, "KD needs enc_full.pt (train `full` first)"
        teacher = PCREncoder(load_phase1_lm()); teacher.load_state_dict(torch.load(tp, map_location="cpu")); teacher.to(DEVICE).eval()
        for p in teacher.parameters(): p.requires_grad = False
        enc = PCREncoder(load_phase1_lm(KD_LAYERS))
        print("teacher params %.1fM -> student params %.1fM" % (count_params(teacher) / 1e6, count_params(enc) / 1e6))
    else:
        enc = PCREncoder(load_phase1_lm(), n_prompt=N_PROMPT if TECHNIQUE == "prompt" else 0)
    enc.to(DEVICE); params = apply_technique(enc, TECHNIQUE); enc.to(DEVICE)
    n_total, n_train = count_params(enc), sum(p.numel() for p in params)
    print(f"[{TECHNIQUE}] total params {n_total / 1e6:.2f}M | trainable {n_train / 1e6:.3f}M ({100 * n_train / n_total:.2f}%)")

    steps, train_secs = 0, 0.0
    if DEVICE.type == "cuda": torch.cuda.reset_peak_memory_stats()
    if TECHNIQUE != "zeroshot": steps, train_secs, _ = run_train(enc, params, LR[TECHNIQUE], teacher)
    peak_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0
    print(f"[{TECHNIQUE}] trained {steps} steps in {train_secs / 60:.1f} min, peak GPU mem {peak_gb:.1f} GB")
    if TECHNIQUE in ("full", "partial", "kd"): torch.save(enc.state_dict(), f"{OUT}/enc_{TECHNIQUE}.pt")

    t0 = time.time(); dev_res, E = evaluate_split(enc, "dev"); infer_secs = time.time() - t0
    print(f"[{TECHNIQUE}] DEV", json.dumps({k: {m: round(x, 4) for m, x in v.items()} for k, v in dev_res.items()}))
    pool_choice = POOLING if POOLING != "auto" else max(dev_res, key=lambda k: dev_res[k]["F1@7"])
    tf_alpha = 0.0
    if USE_TFIDF_FUSION:
        qk, qf, qm, ck, cf, cm = E
        T = tfidf_matrix([SPL["dev"]["q"][q]["text"] for q in qk], [SPL["dev"]["c"][c]["text"] for c in ck])
        Q, C = (qm, cm) if pool_choice == "mean" else (qf, cf); gold = gold_of("dev"); best = (-1, 0)
        for a in [0, 0.25, 0.5, 1, 2, 4, 8, 1e6]:
            f = micro_f1_at_k(topk_preds(Q @ C.T + a * T, qk, ck), gold)["F1@7"]
            if f > best[0]: best = (f, a)
        tf_alpha = best[1]

    tq_k, tq_f, tq_m = embed_docs(enc, TOK["test"]["q"]); tc_k, tc_f, tc_m = embed_docs(enc, TOK["test"]["c"])
    Q, C = (tq_m, tc_m) if pool_choice == "mean" else (tq_f, tc_f); S = Q @ C.T
    if USE_TFIDF_FUSION and tf_alpha:
        S = S + tf_alpha * tfidf_matrix([SPL["test"]["q"][q]["text"] for q in tq_k], [SPL["test"]["c"][c]["text"] for c in tc_k])
    pred = topk_preds(S, tq_k, tc_k)
    pool_ids = set(SPL["test"]["c"]); assert set(pred) == set(SPL["test"]["q"]), "missing/extra test queries"
    for q, p in pred.items(): assert len(p) == 7 and len(set(p)) == 7 and all(c in pool_ids for c in p), (q, p)
    pj = f"{OUT}/prediction_{TECHNIQUE}.json"; json.dump(pred, open(pj, "w"), indent=1)
    zp = f"{OUT}/submission_{TECHNIQUE}.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf: zf.write(pj, arcname="prediction.json")
    print(f"[{TECHNIQUE}] WROTE {zp} | queries: {len(pred)} | pooling={pool_choice}")

    row = {"technique": TECHNIQUE, "pooling": pool_choice, "dev_F1@7_mean": round(dev_res["mean"]["F1@7"], 4),
           "dev_F1@7_first": round(dev_res["first"]["F1@7"], 4), "dev_P@7": round(dev_res[pool_choice]["P@7"], 4),
           "dev_R@7": round(dev_res[pool_choice]["R@7"], 4), "total_params_M": round(n_total / 1e6, 2),
           "trainable_params_M": round(n_train / 1e6, 3), "steps": steps, "train_min": round(train_secs / 60, 1),
           "model_mem_MB": round((sum(p.numel() * p.element_size() for p in enc.parameters()) + sum(b.numel() * b.element_size() for b in enc.buffers())) / 1e6, 1),
           "peak_gpu_GB": round(peak_gb, 2), "dev_embed_secs": round(infer_secs, 1), "fusion_alpha": tf_alpha,
           "total_run_min": round((time.time() - t_start) / 60, 1)}
    rp = f"{OUT}/pcr_results.csv"; new = not os.path.exists(rp)
    with open(rp, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(row)); (w.writeheader() if new else None); w.writerow(row)
    print(row)
    del enc, teacher, params; gc.collect()
    if DEVICE.type == "cuda": torch.cuda.empty_cache()
    return row

done, failed, t_all = {}, {}, time.time()
for tech in TECH_LIST:
    if os.path.exists(f"{OUT}/submission_{tech}.zip") and tech != "full":
        print("skip (zip already exists):", tech); continue
    print("\n" + "=" * 20, tech, "| elapsed %.0f min" % ((time.time() - t_all) / 60), "=" * 20)
    try: done[tech] = run_technique(tech)
    except Exception:
        failed[tech] = traceback.format_exc(); print("!!! FAILED", tech); print(failed[tech])
        gc.collect(); torch.cuda.empty_cache() if DEVICE.type == "cuda" else None

print("\nFINISHED in %.0f min" % ((time.time() - t_all) / 60))
print("OK    :", {t: r["dev_F1@7_" + r["pooling"]] for t, r in done.items()})
print("FAILED:", list(failed))
print("zips  :", sorted(f for f in os.listdir(OUT) if f.endswith(".zip")))


==================== full | elapsed 0 min ====================
[full] total params 109.51M | trainable 109.505M (100.00%)
step 20 t=25s lr=2.66e-05 loss=2.7057 {'ce': 2.7057}
step 40 t=49s lr=5.00e-05 loss=1.7473 {'ce': 1.7473}
step 60 t=75s lr=4.99e-05 loss=2.4200 {'ce': 2.42}
step 80 t=102s lr=4.95e-05 loss=1.3421 {'ce': 1.3421}
step 100 t=129s lr=4.89e-05 loss=2.8693 {'ce': 2.8693}
step 120 t=155s lr=4.80e-05 loss=2.1179 {'ce': 2.1179}
step 140 t=182s lr=4.69e-05 loss=1.9033 {'ce': 1.9033}
step 160 t=209s lr=4.56e-05 loss=3.2457 {'ce': 3.2457}
step 180 t=236s lr=4.42e-05 loss=2.2927 {'ce': 2.2927}
step 200 t=263s lr=4.25e-05 loss=2.5505 {'ce': 2.5505}
step 220 t=290s lr=4.06e-05 loss=2.7257 {'ce': 2.7257}
step 240 t=317s lr=3.86e-05 loss=1.8224 {'ce': 1.8224}
step 260 t=344s lr=3.65e-05 loss=2.1666 {'ce': 2.1666}
step 280 t=371s lr=3.42e-05 loss=2.0880 {'ce': 2.088}
step 300 t=398s lr=3.19e-05 loss=2.4695 {'ce': 2.4695}
step 320 t=425s lr=2.95e-05 loss=2.3638 {'ce': 2.3638}
step 34